In [ ]:
import sys
!pip install --upgrade "figuard[langchain]>=0.3.0" langchain-openai -q
for _mod in list(sys.modules.keys()):
    if _mod.startswith("figuard"):
        del sys.modules[_mod]
import figuard
print(f"✓ FiGuard {figuard.__version__} + LangChain ready")

In [ ]:
from langchain_openai import ChatOpenAI
from figuard.integrations.langchain import FiGuardCallbackHandler
from figuard import FiGuardClient

client = FiGuardClient(
    api_key="sb_live_demo",
    base_url="https://figuard-sandbox-1.onrender.com",
)

budget = client.create_budget(
    user_id="agent_001",
    total_limit=500.00,
    currency="USD",
    expires_in="24h",
    intent_context="shopping assistant session",
)

print(f"✓ Budget created: {budget.id}")
print(f"  Total limit: ${budget.total_limit}")

# Extract session token from tokens array (one entry per dimension)
tokens = {t.category: t.session_token for t in budget.tokens}
session_token = tokens["default"]  # simple budget uses "default" category

In [ ]:
# Wire FiGuard into the LangChain callback stack
# Every tool call is pre-flight authorized before it executes
import os

handler = FiGuardCallbackHandler(
    client=client,
    session_token=session_token,
    agent_id="shopping_agent",
)

# OPENAI_API_KEY must be set for the LLM to run
# os.environ["OPENAI_API_KEY"] = "your-key-here"

llm = ChatOpenAI(callbacks=[handler])
# Every tool call the LLM makes is now gated through FiGuard
# Denied calls raise FiGuardDeniedError before the tool executes

print("✓ LangChain + FiGuard wired up")
print("  Set OPENAI_API_KEY and invoke your chain to see live enforcement.")
print(f"\n→ See the spend tree: https://figuard-sandbox-1.onrender.com/ui")